****Toxic Comment Moderation Classifer****

**Problem Statement**
Online news platforms rely on human moderators to review reader comments, a process that doesn't scale as comment volume grows. This project builds an automated moderation pipeline that classifies toxic comments using machine learning, with an LLM-assisted routing layer for cases the 
classifier is uncertain about.

**Dataset:** Civil Comments (Jigsaw Unintended Bias in Toxicity Classification) -- 91K real news-site comments with crowd-sourced toxicity scores, moderator decisions, and identity-group annotations.

**Target:** `is_toxic` binarized from the crowd toxicity score at threshold 0.5. Chosen over the moderator rejection label after EDA revealed that publication-level policy differences account for up to a 5x variation in rejection rate at comparable toxicity levels, making rejection prediction text-based but with a hard ceiling from information not in the comment itself.

**Architecture:** XGBoost classifier handles high-confidence predictions.Cases where the model's predicted probability falls in the 0.4–0.8 uncertainty band are routed to Gemini for semantic reasoning. The combined system improves F1 from 0.854 to 0.878 while sending under 5% of traffic to the LLM.

**1. Exploratory Data Analysis**

The dataset offers two candidate targets: `target` (a continuous crowd 
toxicity score) and `rating` (the moderator's approve/reject decision). 
Before committing to either, I investigated how they relate and what 
drives moderation outcomes — starting with sub-score breakdowns by 
rating, then toxicity bands, then publication-level effects.

In [1]:
import pandas as pd, numpy as np, re, json
import matplotlib.pyplot as plt, seaborn as sns
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())

df = pd.read_csv("C:/Users/LENOVO/Downloads/data.csv")
print("shape:", df.shape)

# two candidate targets
df['is_toxic']    = (df['target'] >= 0.5).astype(int)
df['is_rejected'] = (df['rating'] == 'rejected').astype(int)

print("\ntoxicity (crowd label):"); print(df['is_toxic'].value_counts())
print("\nrejection (moderator decision):"); print(df['is_rejected'].value_counts())
print("\nrejection rate:", round(df['is_rejected'].mean()*100, 1), "%")

shape: (90902, 45)

toxicity (crowd label):
is_toxic
1    45451
0    45451
Name: count, dtype: int64

rejection (moderator decision):
is_rejected
0    74917
1    15985
Name: count, dtype: int64

rejection rate: 17.6 %


In [2]:
q = """
WITH stats AS (
    SELECT rating,
           COUNT(*) AS n,
           ROUND(AVG(target), 3)          AS avg_toxicity,
           ROUND(AVG(insult), 3)          AS avg_insult,
           ROUND(AVG(threat), 3)          AS avg_threat,
           ROUND(AVG(identity_attack), 3) AS avg_identity_attack,
           ROUND(AVG(obscene), 3)         AS avg_obscene
    FROM df GROUP BY rating
)
SELECT *, ROUND(100.0 * n / (SELECT SUM(n) FROM stats), 1) AS pct
FROM stats ORDER BY avg_toxicity DESC;
"""
pysqldf(q)

,rating,n,avg_toxicity,avg_insult,avg_threat,avg_identity_attack,avg_obscene,pct
0,rejected,15985,0.725,0.646,0.049,0.102,0.185,17.6
1,approved,74917,0.367,0.325,0.021,0.045,0.072,82.4


In [3]:
q = """
WITH bucketed AS (
    SELECT CASE
             WHEN target < 0.2 THEN '0.0-0.2'
             WHEN target < 0.5 THEN '0.2-0.5'
             WHEN target < 0.8 THEN '0.5-0.8'
             ELSE '0.8-1.0' END AS toxicity_band,
           rating,
           LENGTH(comment_text) AS len
    FROM df
)
SELECT toxicity_band,
       COUNT(*) AS n,
       SUM(CASE WHEN rating='rejected' THEN 1 ELSE 0 END) AS rejected,
       ROUND(100.0 * SUM(CASE WHEN rating='rejected' THEN 1 ELSE 0 END) / COUNT(*), 1) AS reject_rate,
       ROUND(AVG(len), 0) AS avg_length
FROM bucketed
GROUP BY toxicity_band
ORDER BY toxicity_band;
"""
pysqldf(q)

,toxicity_band,n,rejected,reject_rate,avg_length
0,0.0-0.2,42134,1837,4.4,293.0
1,0.2-0.5,3317,318,9.6,334.0
2,0.5-0.8,14620,4614,31.6,195.0
3,0.8-1.0,30831,9216,29.9,217.0


In [4]:
q = """
SELECT publication_id,
       COUNT(*) AS n,
       ROUND(AVG(target),3) AS avg_toxicity,
       ROUND(100.0*SUM(CASE WHEN rating='rejected' THEN 1 ELSE 0 END)/COUNT(*),1) AS reject_rate
FROM df
GROUP BY publication_id
HAVING n > 500
ORDER BY reject_rate DESC;
"""
pysqldf(q)

,publication_id,n,avg_toxicity,reject_rate
0,43,608,0.496,34.2
1,105,3456,0.631,28.1
2,6,597,0.479,23.1
3,54,30130,0.406,22.7
4,102,11808,0.467,18.3
5,21,20832,0.472,16.3
6,22,2156,0.279,12.2
7,100,1155,0.456,11.7
8,55,7252,0.466,10.2
9,13,7165,0.391,8.5


In [5]:
identity_cols = ['asian','black','christian','female','male','muslim','jewish',
                 'homosexual_gay_or_lesbian','white','psychiatric_or_mental_illness']

sub = df[df[identity_cols].notna().all(axis=1)].copy()
print("rows with identity annotations:", len(sub))
baseline = sub['is_toxic'].mean()

rows = []
for c in identity_cols:
    m = sub[sub[c] > 0.5]
    if len(m) > 50:
        rows.append({'identity': c, 'n': len(m),
                     'toxicity_rate': round(m['is_toxic'].mean(), 3)})
bias = pd.DataFrame(rows).sort_values('toxicity_rate', ascending=False)
bias['baseline'] = round(baseline, 3)
bias['lift'] = (bias['toxicity_rate'] / baseline).round(2)
print(bias.to_string(index=False))

rows with identity annotations: 21687
                     identity    n  toxicity_rate  baseline  lift
                        black 1160          0.842     0.565  1.49
                        white 1692          0.803     0.565  1.42
    homosexual_gay_or_lesbian  728          0.801     0.565  1.42
psychiatric_or_mental_illness  321          0.757     0.565  1.34
                       muslim 1301          0.737     0.565  1.31
                       jewish  368          0.685     0.565  1.21
                         male 2228          0.627     0.565  1.11
                       female 2613          0.586     0.565  1.04
                        asian  177          0.525     0.565  0.93
                    christian 1354          0.396     0.565  0.70


In [6]:
import nltk, re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean(text):
    text = re.sub(r'http\S+|www\.\S+', ' ', str(text).lower())
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = nltk.word_tokenize(text)
    return " ".join(stemmer.stem(w) for w in tokens
                    if w not in stop_words and len(w) > 2)

df['clean_text'] = df['comment_text'].apply(clean)
df = df[df['clean_text'].str.strip() != ""]      # drop rows emptied by cleaning
print("rows after cleaning:", len(df))
print(df[['comment_text','clean_text']].head(3).to_string())

rows after cleaning: 90699
                                                                                                                                                                                           comment_text                                                                                clean_text
0                                                                                                                                                                  haha you guys are a bunch of losers.                                                                      haha guy bunch loser
1  Yet call out all Muslims for the acts of a few will get you pilloried.   So why is it okay to smear an entire religion over these few idiots?  Or is this because it's okay to bash Christian sects?  yet call muslim act get pillori okay smear entir religion idiot okay bash christian sect
2                                                                                                      

In [7]:
from sklearn.model_selection import train_test_split

# deliberately downsample positives to ~15% to reflect a realistic moderation queue
neg = df[df.is_toxic == 0]
pos = df[df.is_toxic == 1].sample(n=int(len(neg) * 0.15 / 0.85), random_state=42)
data = pd.concat([neg, pos]).sample(frac=1, random_state=42).reset_index(drop=True)

print("class balance:"); print(data.is_toxic.value_counts(normalize=True).round(3))

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    data['clean_text'], data['is_toxic'],
    test_size=0.2, random_state=42, stratify=data['is_toxic'])
print("train:", X_train_txt.shape, "| test:", X_test_txt.shape)

class balance:
is_toxic
0    0.85
1    0.15
Name: proportion, dtype: float64
train: (42593,) | test: (10649,)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
X_train = tfidf.fit_transform(X_train_txt)
X_test  = tfidf.transform(X_test_txt)
print("feature matrix:", X_train.shape)

feature matrix: (42593, 20000)


In [9]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve, f1_score, classification_report

spw = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(spw, 2))

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(max_depth=30, min_samples_leaf=5,
                                            class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                            class_weight='balanced_subsample',
                                            n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1,
                             subsample=0.8, colsample_bytree=0.8,
                             scale_pos_weight=spw, eval_metric='logloss',
                             n_jobs=-1, random_state=42),
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    prob = m.predict_proba(X_test)[:, 1]

    f1_default = f1_score(y_test, (prob >= 0.5).astype(int), zero_division=0)
    p, r, t = precision_recall_curve(y_test, prob)
    f1s = 2*p*r/(p+r+1e-9)
    i = np.nanargmax(f1s)
    best_t = t[i] if i < len(t) else 0.5

    results[name] = {'model': m, 'prob': prob, 'threshold': best_t, 'f1': f1s[i]}
    print(f"\n{name}")
    print(f"  F1 @0.5   : {f1_default:.3f}")
    print(f"  Best F1   : {f1s[i]:.3f}  (threshold {best_t:.3f})")
    print(f"  P / R     : {p[i]:.3f} / {r[i]:.3f}")

scale_pos_weight: 5.67

Logistic Regression
  F1 @0.5   : 0.833
  Best F1   : 0.841  (threshold 0.576)
  P / R     : 0.869 / 0.815

Decision Tree
  F1 @0.5   : 0.776
  Best F1   : 0.779  (threshold 0.618)
  P / R     : 0.833 / 0.731

Random Forest
  F1 @0.5   : 0.823
  Best F1   : 0.825  (threshold 0.499)
  P / R     : 0.846 / 0.804

XGBoost
  F1 @0.5   : 0.850
  Best F1   : 0.854  (threshold 0.630)
  P / R     : 0.885 / 0.825


In [10]:
best = results["XGBoost"]
prob, thr = best['prob'], best['threshold']

for lo, hi in [(0.3, 0.8), (0.4, 0.8), (0.35, 0.75)]:
    band = (prob >= lo) & (prob <= hi)
    pred = (prob >= thr).astype(int)
    correct = (pred[band] == y_test.values[band]).mean() if band.sum() else 0
    print(f"band {lo}-{hi}: {band.sum():5d} cases ({band.mean()*100:4.1f}%) | "
          f"XGB accuracy inside band: {correct:.3f} | "
          f"positives inside: {y_test.values[band].sum()}")

print(f"\nOverall XGB accuracy: {((prob>=thr).astype(int) == y_test.values).mean():.3f}")

band 0.3-0.8:   777 cases ( 7.3%) | XGB accuracy inside band: 0.725 | positives inside: 273
band 0.4-0.8:   510 cases ( 4.8%) | XGB accuracy inside band: 0.651 | positives inside: 237
band 0.35-0.75:   557 cases ( 5.2%) | XGB accuracy inside band: 0.679 | positives inside: 204

Overall XGB accuracy: 0.958


XGBoost's overall accuracy is 0.958. Inside the 0.4–0.8 probability 
band it drops to 0.651, the cases where TF-IDF features are 
insufficient and semantic reasoning adds value. This band (4.8% of 
traffic) is what gets routed to Gemini.

In [ ]:
# Save trained model and vectorizer for the FastAPI service
import joblib
joblib.dump({'xgb': models['XGBoost'], 'tfidf': tfidf}, 'model.pkl')
print("Saved model.pkl")

In [ ]:
import os, json, time
os.environ["GEMINI_API_KEY"] 

import google.generativeai as genai
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {"toxic_flag": {"type": "string", "enum": ["yes", "no"]}},
    "required": ["toxic_flag"]
}

SYSTEM_PROMPT = """You are a content moderation classifier for online news comments.

Given a comment, decide whether a human moderator would consider it toxic.

Mark "yes" if the comment contains insults, personal attacks, obscenity,
threats, identity-based attacks, or hostile language likely to make someone
leave the discussion. Mark "yes" even for mild name-calling, dismissive
insults, or condescending remarks aimed at people rather than ideas.

Mark "no" for ordinary opinions, disagreement, criticism of ideas or
policies, and strongly worded but civil argument.

Return JSON matching the required schema."""

GEN_CONFIG = {
    "temperature": 0.0,
    "top_k": 1,
    "top_p": 1.0,
    "max_output_tokens": 2048,
    "response_mime_type": "application/json",
    "response_schema": RESPONSE_SCHEMA,
}

gem = genai.GenerativeModel("gemini-3.1-flash-lite", system_instruction=SYSTEM_PROMPT)

_cache = {}
def classify(text):
    if text in _cache: return _cache[text]
    for attempt in range(3):
        try:
            r = gem.generate_content(text, generation_config=GEN_CONFIG)
            flag = json.loads(r.text).get("toxic_flag", "no")
            _cache[text] = flag
            time.sleep(4)
            return flag
        except Exception as e:
            if "429" in str(e):
                print(f"  rate limit, waiting 20s (attempt {attempt+1})")
                time.sleep(20)
                continue
            print("  ERR:", type(e).__name__, str(e)[:60])
            break
    _cache[text] = "ERROR"
    return "ERROR"

In [14]:
import time
band = (prob >= 0.4) & (prob <= 0.8)
band_texts  = X_test_txt[band].index          # original df indices
band_raw    = data.loc[band_texts, 'comment_text'].tolist()
band_y      = y_test.values[band]

hybrid_pred = (prob >= thr).astype(int).copy()
band_idx    = np.where(band)[0]

for i, (pos, text) in enumerate(zip(band_idx, band_raw)):
    flag = classify(text[:1000])
    if flag != "ERROR":
        hybrid_pred[pos] = 1 if flag == "yes" else 0
    if i % 50 == 0:
        print(f"{i}/{len(band_raw)}")

print("\nXGBoost alone :", f1_score(y_test, (prob>=thr).astype(int)).round(3))
print("Hybrid        :", f1_score(y_test, hybrid_pred).round(3))

0/510
50/510
100/510
150/510
200/510
250/510
300/510
350/510
400/510
450/510
  rate limit, waiting 20s (attempt 1)
  rate limit, waiting 20s (attempt 2)
  rate limit, waiting 20s (attempt 3)
500/510
  rate limit, waiting 20s (attempt 1)
  rate limit, waiting 20s (attempt 2)
  rate limit, waiting 20s (attempt 3)
  rate limit, waiting 20s (attempt 1)
  rate limit, waiting 20s (attempt 2)
  rate limit, waiting 20s (attempt 3)
  rate limit, waiting 20s (attempt 1)
  rate limit, waiting 20s (attempt 2)
  rate limit, waiting 20s (attempt 3)
  rate limit, waiting 20s (attempt 1)
  rate limit, waiting 20s (attempt 2)
  rate limit, waiting 20s (attempt 3)
  rate limit, waiting 20s (attempt 1)


KeyboardInterrupt: 

In [15]:
import pickle, joblib
with open('gemini_cache.pkl', 'wb') as f:
    pickle.dump(_cache, f)
joblib.dump(hybrid_pred, 'hybrid_pred_partial.pkl')
print(f"saved. cache entries: {len(_cache)}")

saved. cache entries: 505


In [16]:
print("ERROR count in cache:", sum(v == "ERROR" for v in _cache.values()))
print("hybrid_pred vs xgb alone:")
print("XGBoost alone:", f1_score(y_test, (prob>=thr).astype(int)).round(3))
print("Hybrid (505/510):", f1_score(y_test, hybrid_pred).round(3))

ERROR count in cache: 5
hybrid_pred vs xgb alone:
XGBoost alone: 0.854
Hybrid (505/510): 0.878
